In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import cvxpy as cp
from itertools import combinations
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm
from matplotlib.ticker import MultipleLocator

In [ ]:
def smce_hat(f_X, Y):
    # Using smCE version of calculating dCE
    n = len(f_X)
    sorted_indices = np.argsort(f_X)
    Y = Y[sorted_indices]
    f_X = f_X[sorted_indices]
    g = cp.Variable(n)
    objective = cp.Maximize(cp.sum(cp.multiply(Y - f_X, g)) / n)
    constraints = [cp.abs(g[i+1] - g[i]) <= cp.abs(f_X[i+1] - f_X[i]) for i in range(n-1)]
    constraints += [g >= -1, g <= 1]
    prob = cp.Problem(objective, constraints)
    sup_val = prob.solve()

    return sup_val

In [ ]:
def cutoff_hat(f_X, Y):
    n = len(f_X)
    sorted_fX = np.sort(f_X)
    supremum = -np.inf
    best_interval = None

    for i, a in enumerate(sorted_fX):
        for b in sorted_fX[i:]:
            mask = (f_X >= a) & (f_X <= b)
            value = np.abs(np.mean((Y - f_X) * mask))

            if value > supremum:
                supremum = value
                best_interval = (a, b)

    return supremum

In [ ]:
def get_bins(f_X, m):
  bins = np.ceil(f_X *m).astype(int)
  bins = np.clip(bins, 1, m)
  return bins

In [ ]:
def c_cdl_m(f_X, Y, m):
  n = len(f_X)
  w = np.zeros((m+1,n))
  pi = np.zeros(m+1) #0-indexed
  q =  np.zeros(m+1) #0-indexed

  for i in range(m+1):
    for j in range(n):
      if 1- abs(m*f_X[j]-i) > 0:
        w[i,j] = 1 - abs(m*f_X[j]-i)
    pi[i] = np.mean(w[i,])
    if pi[i]> 0:
      q[i] = np.mean(w[i,] * Y) / pi[i]

  best = 0
  for i in range(m+1):
    val = 0
    for j in range(m+1):
      if j<= i:
        if q[j] - 1.0*(i+1)/m > 0:
          val += pi[j] * (q[j] - 1.0*(i+1)/m)
      else:
        if 1.0*i/m - q[j] > 0:
          val += pi[j] * (1.0*i/m-q[j])
    best = max(best, val)

  return best

def scdl(f_X, Y, max_power=24):
  best = np.inf
  m_best = 1
  lo, hi = 1, max_power
  while lo < hi:
    mid = (lo + hi) // 2
    m = 2 ** mid
    if c_cdl_m(f_X, Y, m) < 1.0 / m:
      lo = mid + 1
    else:
      hi = mid

  m_best = 2 ** lo
  best = min(max(c_cdl_m(f_X, Y, m_best), 1.0 / m_best), 2.0/m_best)

  return best, m_best

In [ ]:
def ece_hat(f_X, Y, m):
  n = len(f_X)
  bins = get_bins(f_X, m)
  pi = np.zeros(m) #0-indexed
  q_star =  np.zeros(m) #0-indexed
  q = np.zeros(m)

  for i in range(m):
    mask = bins == (i + 1)
    pi[i] = np.mean(mask)
    if pi[i]> 0:
      q_star[i] = np.mean(Y[mask])
      q[i] = np.mean(f_X[mask])

  return np.sum(pi * np.abs(q_star-q))


In [ ]:
def ece(f_X_unique, E_Y_f_X_unique):
  return np.mean(np.abs(f_X_unique - E_Y_f_X_unique))

In [ ]:
def risk(f_X_unique, E_Y_f_X_unique, tau):
  return tau * np.mean((1-E_Y_f_X_unique) * (f_X_unique >= tau)) + (1-tau) * np.mean(E_Y_f_X_unique * (f_X_unique < tau))

def risk_indicator(indicator_guess, E_Y_f_X_unique, tau):
  return tau * np.mean((1-E_Y_f_X_unique) * indicator_guess) + (1-tau) * np.mean(E_Y_f_X_unique * (1-indicator_guess))

# Prediction Curves

In [ ]:
np.random.seed(1)
n = 500 #number of training samples
N = 1000 #number of calibration samples


X = np.random.uniform(size=n)
X_population = np.random.uniform(size=N)

alpha = 0.8

E_Y_X = alpha*(1 - 4*X + 4 * X**2) + (1-alpha)*X
E_Y_X_population = alpha*(1 - 4*X_population + 4 * X_population**2) + (1-alpha)*X_population

Y = np.random.binomial(n=1, p=E_Y_X)
X_model = sm.add_constant(X)
X_population_model = sm.add_constant(X_population)
model = sm.Logit(Y, X_model).fit(disp=0)
f_X = model.predict(X_model)
f_X_population = model.predict(X_population_model)


sorted_idx = np.argsort(X_population)
X_sorted = X_population[sorted_idx]
E_Y_sorted = E_Y_X_population[sorted_idx]
f_X_sorted = f_X_population[sorted_idx]

plt.figure(figsize=(8, 5))
plt.plot(X_sorted, E_Y_sorted, label=r'$E[Y|f(X)]$')
plt.plot(X_sorted, f_X_sorted, label=r'$f(X)$', linestyle='--')
plt.plot(X_sorted, 1 - f_X_sorted, label=r'$1 - f(X)$', linestyle=':')
plt.xlabel('X')
plt.ylabel('Value')
plt.text(0.05, 0.95, rf'$\alpha = {alpha}$', transform=plt.gca().transAxes,
         fontsize=12, verticalalignment='top')
plt.legend()
plt.show()


# Actionability

In [ ]:
np.random.seed(1)
n = 500 #number of training samples
N = 1000 #number of calibration samples
S = 200 #number of iterations
tau = 0.35 #threshold for risk

df = pd.DataFrame(columns=['smce','cutoff','ece', 'scdl', 'm', 'risk', 'risk_oracle', 'alpha'])

for s in tqdm(range(S)):
  X = np.random.uniform(size=n)
  X_population = np.random.uniform(size=N)

  alpha = np.random.uniform(size=1)

  E_Y_X = alpha*(1 - 4*X + 4 * X**2) + (1-alpha)*X
  E_Y_X_population = alpha*(1 - 4*X_population + 4 * X_population**2) + (1-alpha)*X_population

  Y = np.random.binomial(n=1, p=E_Y_X)
  X_model = sm.add_constant(X)
  X_population_model = sm.add_constant(X_population)
  model = sm.Logit(Y, X_model).fit(disp=0)
  f_X = 1- model.predict(X_model)
  f_X_population = 1-model.predict(X_population_model)

  df.loc[s, 'smce'] = smce_hat(f_X_population, E_Y_X_population)
  df.loc[s, 'cutoff'] = cutoff_hat(f_X_population, E_Y_X_population)
  df.loc[s, 'ece'] = ece(f_X_population, E_Y_X_population)
  df.loc[s, 'scdl'], df.loc[s,'m'] = scdl(f_X_population, E_Y_X_population)
  df.loc[s, 'risk'] = risk(f_X_population, E_Y_X_population, tau)
  df.loc[s, 'risk_oracle'] = risk(E_Y_X_population, E_Y_X_population, tau)
  df.loc[s, 'alpha'] = alpha


df['swap_regret'] = df['risk'] - df['risk_oracle']

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6), sharey='all')

x_min = -0.01
x_max = max(df['smce'].max(), df['cutoff'].max(), df['ece'].max(), df['scdl'].max()) * 1.1

props = dict(boxstyle='round', facecolor='wheat', alpha=0.9)


axes[0].scatter(df['smce'], df['swap_regret'], c=df['alpha'], cmap='Blues', edgecolor='k', alpha=0.75, s = 4*mpl.rcParams['lines.markersize']**2)
axes[0].set_xlabel(r'$\mathsf{smCE}$', fontsize=30)
axes[0].set_ylabel('Swap Regret', fontsize=30)
axes[0].set_xlim(x_min, x_max)
axes[0].set_ylim(x_min, x_max)
correlation = df.corr(method='spearman').loc['smce','swap_regret']
axes[0].annotate(f'Spearman Correlation: {correlation: 0.02f}', xy=(0.05, 0.9), xycoords='axes fraction', fontsize=18, bbox=props)

axes[1].scatter(df['cutoff'], df['swap_regret'],  c=df['alpha'], cmap='Blues', edgecolor='k', alpha=0.75, s = 4*mpl.rcParams['lines.markersize']**2)
axes[1].set_xlabel(r'$\mathsf{Cutoff}$', fontsize=30)
axes[1].set_xlim(x_min, x_max)
correlation = df.corr(method='spearman').loc['cutoff','swap_regret']
axes[1].annotate(f'Spearman Correlation: {correlation: 0.02f}', xy=(0.05, 0.9), xycoords='axes fraction', fontsize=18, bbox=props)

sc = axes[2].scatter( df['ece'], df['swap_regret'], c=df['alpha'], cmap='Blues', edgecolor='k', alpha=0.75, s = 4*mpl.rcParams['lines.markersize']**2)
line_values = np.append(df['ece'].sort_values(ascending=False).values, 0)
axes[2].set_xlabel(r'$\mathsf{ECE}$', fontsize=30)
axes[2].set_xlim(x_min, x_max)
correlation = df.corr(method='spearman').loc['ece','swap_regret']
axes[2].annotate(f'Spearman Correlation: {correlation: 0.02f}', xy=(0.05, 0.9), xycoords='axes fraction', fontsize=18, bbox=props)

axes[3].scatter(df['scdl'], df['swap_regret'], c=df['alpha'], cmap='Blues', edgecolor='k', alpha=0.75, s = 4*mpl.rcParams['lines.markersize']**2)
axes[3].set_xlabel(r'$\mathsf{SCDL}$', fontsize=30)
axes[3].set_xlim(x_min, x_max)
axes[3].set_ylim(x_min, x_max)
correlation = df.corr(method='spearman').loc['scdl','swap_regret']
axes[3].annotate(f'Spearman Correlation: {correlation: 0.02f}', xy=(0.05, 0.9), xycoords='axes fraction', fontsize=18, bbox=props)

fig.subplots_adjust(top=0.75)
cbar_ax = fig.add_axes([0.2, 0.88, 0.6, 0.02])
cbar = fig.colorbar(sc, cax=cbar_ax, orientation='horizontal')
cbar.set_label(r'$\alpha$', fontsize=30, labelpad=-10)
cbar.ax.tick_params(labelsize=24)
sc.set_clim(0, 1)

[t.tick_params(labelsize=16) for t in axes.reshape(-1)]

plt.show()

# DARKER BLUES CORRESPOND TO LARGER ALPHAS/TS

In [ ]:
np.random.seed(1)
n = 500 #number of training samples
N = 1000 #number of calibration samples
S = 200 #number of iterations
tau = 0.35 #threshold for risk

df = pd.DataFrame(columns=['smce','cutoff','ece', 'scdl', 'm', 'risk', 'risk_oracle', 'alpha'])

for s in tqdm(range(S)):
  X = np.random.uniform(size=n)
  X_population = np.random.uniform(size=N)

  alpha = np.random.uniform(size=1)

  E_Y_X = alpha*(1 - 4*X + 4 * X**2) + (1-alpha)*X
  E_Y_X_population = alpha*(1 - 4*X_population + 4 * X_population**2) + (1-alpha)*X_population

  Y = np.random.binomial(n=1, p=E_Y_X)
  X_model = sm.add_constant(X)
  X_population_model = sm.add_constant(X_population)
  model = sm.Logit(Y, X_model).fit(disp=0)
  f_X = model.predict(X_model)
  f_X_population = model.predict(X_population_model)

  df.loc[s, 'smce'] = smce_hat(f_X_population, E_Y_X_population)
  df.loc[s, 'cutoff'] = cutoff_hat(f_X_population, E_Y_X_population)
  df.loc[s, 'ece'] = ece(f_X_population, E_Y_X_population)
  df.loc[s, 'scdl'], df.loc[s,'m'] = scdl(f_X_population, E_Y_X_population)
  df.loc[s, 'risk'] = risk(f_X_population, E_Y_X_population, tau)
  df.loc[s, 'risk_oracle'] = risk(E_Y_X_population, E_Y_X_population, tau)
  df.loc[s, 'alpha'] = alpha

df['swap_regret'] = df['risk'] - df['risk_oracle']

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6), sharey='all')

x_min = -0.01
x_max = max(df['smce'].max(), df['cutoff'].max(), df['ece'].max(), df['scdl'].max()) * 1.1

props = dict(boxstyle='round', facecolor='wheat', alpha=0.9)


axes[0].scatter(df['smce'], df['swap_regret'], c=df['alpha'], cmap='Blues', edgecolor='k', alpha=0.75, s = 4*mpl.rcParams['lines.markersize']**2)
axes[0].set_xlabel(r'$\mathsf{smCE}$', fontsize=30)
axes[0].set_ylabel('Swap Regret', fontsize=30)
axes[0].set_xlim(x_min, x_max)
axes[0].set_ylim(x_min, x_max)
correlation = df.corr(method='spearman').loc['smce','swap_regret']
axes[0].annotate(f'Spearman Correlation: {correlation: 0.02f}', xy=(0.05, 0.9), xycoords='axes fraction', fontsize=18, bbox=props)

axes[1].scatter(df['cutoff'], df['swap_regret'],  c=df['alpha'], cmap='Blues', edgecolor='k', alpha=0.75, s = 4*mpl.rcParams['lines.markersize']**2)
axes[1].set_xlabel(r'$\mathsf{Cutoff}$', fontsize=30)
axes[1].set_xlim(x_min, x_max)
correlation = df.corr(method='spearman').loc['cutoff','swap_regret']
axes[1].annotate(f'Spearman Correlation: {correlation: 0.02f}', xy=(0.05, 0.9), xycoords='axes fraction', fontsize=18, bbox=props)

sc = axes[2].scatter( df['ece'], df['swap_regret'], c=df['alpha'], cmap='Blues', edgecolor='k', alpha=0.75, s = 4*mpl.rcParams['lines.markersize']**2)
line_values = np.append(df['ece'].sort_values(ascending=False).values, 0)
axes[2].set_xlabel(r'$\mathsf{ECE}$', fontsize=30)
axes[2].set_xlim(x_min, x_max)
correlation = df.corr(method='spearman').loc['ece','swap_regret']
axes[2].annotate(f'Spearman Correlation: {correlation: 0.02f}', xy=(0.05, 0.9), xycoords='axes fraction', fontsize=18, bbox=props)

axes[3].scatter(df['scdl'], df['swap_regret'], c=df['alpha'], cmap='Blues', edgecolor='k', alpha=0.75, s = 4*mpl.rcParams['lines.markersize']**2)
axes[3].set_xlabel(r'$\mathsf{SCDL}$', fontsize=30)
axes[3].set_xlim(x_min, x_max)
axes[3].set_ylim(x_min, x_max)
correlation = df.corr(method='spearman').loc['scdl','swap_regret']
axes[3].annotate(f'Spearman Correlation: {correlation: 0.02f}', xy=(0.05, 0.9), xycoords='axes fraction', fontsize=18, bbox=props)

fig.subplots_adjust(top=0.75)
cbar_ax = fig.add_axes([0.2, 0.88, 0.6, 0.02])
cbar = fig.colorbar(sc, cax=cbar_ax, orientation='horizontal')
cbar.set_label(r'$\alpha$', fontsize=30, labelpad=-10)
cbar.ax.tick_params(labelsize=24)
sc.set_clim(0, 1)

[t.tick_params(labelsize=16) for t in axes.reshape(-1)]

plt.show()

# DARKER BLUES CORRESPOND TO LARGER ALPHAS/TS

# Testability


In [ ]:
np.random.seed(1)
n = 500 #number of training samples
N = 1000 #number of calibration samples
S = 100 #number of iterations

df = pd.DataFrame(columns=['smce', 'smce_hat', 'cutoff', 'cutoff_hat', 'ece', 'ece_hat', 'scdl', 'cal_hat', 'm', 'alpha'])

alpha_set = [0, 0.5, 0.8, 1]
for alpha in alpha_set:
  for s in tqdm(range(S)):
    X = np.random.uniform(size=n)
    X_population = np.random.uniform(size=N)


    E_Y_X = alpha*(1 - 4*X + 4 * X**2) + (1-alpha)*X
    E_Y_X_population = alpha*(1 - 4*X_population + 4 * X_population**2) + (1-alpha)*X_population

    Y = np.random.binomial(n=1, p=E_Y_X)
    Y_population = np.random.binomial(n=1, p=E_Y_X_population)

    X_model = sm.add_constant(X)
    X_population_model = sm.add_constant(X_population)
    model = sm.Logit(Y, X_model).fit(disp=0)
    f_X = model.predict(X_model)
    f_X_population = model.predict(X_population_model)

    df.loc[s, 'smce_hat'] = smce_hat(f_X_population, Y_population)
    df.loc[s, 'cutoff_hat'] = cutoff_hat(f_X_population, Y_population)
    df.loc[s, 'cal_hat'], df.loc[s,'m'] = scdl(f_X_population, Y_population)
    df.loc[s, 'ece_hat'] = ece_hat(f_X_population, Y_population, 10)
    df.loc[s, 'alpha'] = alpha

    print(alpha)
  me = df[['smce_hat', 'cutoff_hat', 'ece_hat', 'cal_hat']].mean()
  std = df[['smce_hat', 'cutoff_hat', 'ece_hat', 'cal_hat']].std()
  print("Mean Absolute Error")
  print(me.map(lambda x: f"{x:.6f}"))
  print("Standard Deviation")
  print(std.map(lambda x: f"{x:.6f}"))
